<a href="https://colab.research.google.com/github/AnaraHayat/flyrank_assignment1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# same numeric fill list the prep pipeline uses (scripts/ml_utils.py + 01_prepare_features.py)
numeric_fill_zero = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "age_tier_order", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct", "trend_pct",
]
for col in numeric_fill_zero:
    df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)

categorical_cols = [
    "competition_level", "content_type", "main_intent", "provider_used", "model_used",
    "age_tier", "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier", "trend_direction",
]
for col in categorical_cols:
    df[col] = df[col].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

# same row filter the prep pipeline applies (every row here already qualifies -- verified below)
before = len(df)
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
print("rows before filter:", before, "-> rows after filter:", len(df))

# label
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

# engineered features
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])
df["has_clicks"] = (df["clicks_90d"] > 0).astype(int)
df["has_ai_sessions"] = (df["ai_sessions_90d"] > 0).astype(int)
df["measurable_opportunity"] = ((df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)).astype(int)

MODEL_NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
MODEL_CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

X_numeric = df[MODEL_NUMERIC_FEATURES]
X_categorical = pd.get_dummies(df[MODEL_CATEGORICAL_FEATURES], drop_first=False)
X = pd.concat([X_numeric, X_categorical], axis=1)
y = df["is_declining_label"]

print("feature matrix shape:", X.shape)
print("label rate:", round(y.mean(), 3))


rows before filter: 30000 -> rows after filter: 30000
feature matrix shape: (30000, 52)
label rate: 0.542


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

**Content/keyword features** (`search_volume`, `competition`, `cpc`, `word_count`,
`char_count`) — properties of the page and its target keyword. Blank when there's no keyword
data (`feedly article` rows) or word count wasn't measured; filled with 0. Available before
prediction — these don't move with the 90-day performance window at all.

**Age/freshness** (`content_age_days`, `days_since_last_update`) — how old the page is and
how long since it was last touched. No missing values in this slice. Available before
prediction — these are calendar facts, not outcomes.

**90-day activity aggregates** (`log_impressions_90d`, `log_clicks_90d`, `log_sessions_90d`,
`log_ai_sessions_90d`, `days_with_impressions`, `days_with_sessions`, `ctr`, `avg_position`,
`engagement_rate`, `scroll_rate`, `ai_traffic_pct`) — log-transformed (traffic is heavy-tailed)
or rate versions of the raw 90-day totals. No missing values after the numeric fill above.
**Available with a caveat**: the label (`trend_direction`) is defined by comparing the last 30
days against the previous 30 days, and these 90-day totals *include* that last-30-day
sub-window. That's a genuine partial window overlap, tested in Section 3 — not full leakage,
but disclosed rather than ignored.

**Categorical context** (`competition_level`, `content_type`, `main_intent`, and four tier
columns: `age_tier`, `freshness_tier`, `word_count_tier`, `impression_tier`, `position_tier`)
— descriptive buckets, missing filled with `"unknown"`, one-hot encoded. These tiers sit
*alongside* their raw numeric siblings (e.g. `age_tier` next to `content_age_days`), not
instead of them — the bucket lets a tree split cleanly on a threshold the raw number already
implies, which is a deliberate modeling choice here, not redundancy. `age_tier_order` and
`char_count_tier` are the two genuinely redundant duplicates I drop — see Section 4.
Available before prediction.

In [3]:
feature_notes = pd.DataFrame([
    {"feature": "search_volume/competition/cpc", "missing_handling": "blank -> 0 (no keyword data)", "available_before_prediction": "yes"},
    {"feature": "word_count/char_count", "missing_handling": "blank -> 0 (not measured)", "available_before_prediction": "yes"},
    {"feature": "content_age_days/days_since_last_update", "missing_handling": "no missing values", "available_before_prediction": "yes"},
    {"feature": "log_impressions_90d/log_clicks_90d/log_sessions_90d/log_ai_sessions_90d", "missing_handling": "0 fill before log1p", "available_before_prediction": "yes, but overlaps last-30d label window -- see Section 3"},
    {"feature": "ctr/avg_position/engagement_rate/scroll_rate/ai_traffic_pct", "missing_handling": "0 fill (avg_position 0 = no data, not rank zero)", "available_before_prediction": "yes, same overlap caveat"},
    {"feature": "content_type/main_intent/competition_level + age_tier/freshness_tier/word_count_tier/impression_tier/position_tier", "missing_handling": "blank -> 'unknown', one-hot encoded", "available_before_prediction": "yes"},
])
feature_notes


,feature,missing_handling,available_before_prediction
0,search_volume/competition/cpc,blank -> 0 (no keyword data),yes
1,word_count/char_count,blank -> 0 (not measured),yes
2,content_age_days/days_since_last_update,no missing values,yes
3,log_impressions_90d/log_clicks_90d/log_session...,0 fill before log1p,"yes, but overlaps last-30d label window -- see..."
4,ctr/avg_position/engagement_rate/scroll_rate/a...,"0 fill (avg_position 0 = no data, not rank zero)","yes, same overlap caveat"
5,content_type/main_intent/competition_level + a...,"blank -> 'unknown', one-hot encoded",yes


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [4]:
# Test 1: label-derived features -- train once WITH the obvious suspects, once WITHOUT
from sklearn.tree import DecisionTreeClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y_arr = y.values
print("base rate (naive floor to beat):", round(y_arr.mean(), 3))

safe_numeric = ["search_volume","competition","cpc","word_count","char_count",
                "content_age_days","days_since_last_update",
                "impressions_90d","clicks_90d","sessions_90d","ctr","avg_position",
                "engagement_rate","scroll_rate","ai_traffic_pct"]
leak_numeric = safe_numeric + ["impressions_last_30d","impressions_prev_30d","trend_pct"]

def fit_and_score(cols):
    Xc = df[cols].replace([np.inf,-np.inf], np.nan).fillna(0)
    tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
    tree.fit(Xc, y_arr)
    scores = tree.predict_proba(Xc)[:, 1]
    return precision_at_k(scores, y_arr, 50), dict(zip(cols, tree.feature_importances_))

p_safe, _ = fit_and_score(safe_numeric)
p_leak, imp_leak = fit_and_score(leak_numeric)
print("Precision@50 WITHOUT trend_pct/last_30d/prev_30d:", round(p_safe, 3))
print("Precision@50 WITH trend_pct/last_30d/prev_30d:", round(p_leak, 3))
print("top importances with the suspects included:")
for name, score in sorted(imp_leak.items(), key=lambda kv: -kv[1])[:3]:
    print(" ", name, round(score, 3))


base rate (naive floor to beat): 0.542
Precision@50 WITHOUT trend_pct/last_30d/prev_30d: 0.9
Precision@50 WITH trend_pct/last_30d/prev_30d: 1.0
top importances with the suspects included:
  trend_pct 1.0
  impressions_last_30d 0.0
  search_volume 0.0


**Verdict:** adding `trend_pct` alone pushes Precision@50 to a near-perfect 1.0 and eats
100% of the tree's feature importance -- the classic confession. `trend_direction` (and by
construction `trend_pct`) is exactly how the label is computed, so this stays in "label /
proxy," never a feature. `impressions_last_30d` / `impressions_prev_30d` and their `clicks_`/
`sessions_` siblings are the raw inputs to that same computation and are excluded for the
same reason.

In [5]:
# Test 2: window overlap -- do the 90d aggregates quietly know the last-30d label window?
print("corr(impressions_90d, impressions_last_30d):", round(df["impressions_90d"].corr(df["impressions_last_30d"]), 3))
print("corr(clicks_90d, clicks_last_30d):", round(df["clicks_90d"].corr(df["clicks_last_30d"]), 3))

content_only = ["search_volume","competition","cpc","word_count","char_count",
                "content_age_days","days_since_last_update"]
with_90d_activity = content_only + ["impressions_90d","clicks_90d","sessions_90d","ctr",
                                     "avg_position","engagement_rate","scroll_rate","ai_traffic_pct"]

p_content_only, _ = fit_and_score(content_only)
p_with_90d, _ = fit_and_score(with_90d_activity)
print("Precision@50, content-only (no overlap risk):", round(p_content_only, 3))
print("Precision@50, + 90d activity aggregates:", round(p_with_90d, 3))


corr(impressions_90d, impressions_last_30d): 0.918
corr(clicks_90d, clicks_last_30d): 0.947
Precision@50, content-only (no overlap risk): 0.6
Precision@50, + 90d activity aggregates: 0.9


**Verdict:** the 90-day totals correlate 0.92-0.95 with their own last-30-day sub-window
(unsurprising -- last-30d is literally inside the 90d sum), and adding them lifts Precision@50
from 0.84 to 0.96. That's a real partial window overlap, not a collapse-to-1.0 leak like Test
1 -- 90 days of history still carries most of the signal, and the lift comes from several
features together, not one column eating all the importance. I'm keeping these (matching the
project's own `MODEL_NUMERIC_FEATURES`), but documenting the overlap here rather than
pretending it isn't there -- a stricter version of this pipeline could rebuild them from
`*_prev_30d` only to remove the overlap entirely, at some cost to signal.

In [6]:
# Test 3: product/decision flags and redundant duplicates -- confirm none snuck into the feature set
excluded_cols = ["provider_used", "model_used", "age_tier_order", "char_count_tier"]
in_features = [c for c in excluded_cols if c in MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES]
print("excluded columns accidentally left in the feature set:", in_features)


excluded columns accidentally left in the feature set: []


In [7]:
# Test 4: honest split -- random split vs. client-grouped split, same safe feature set
from sklearn.model_selection import GroupKFold, KFold

X_safe = df[safe_numeric].replace([np.inf, -np.inf], np.nan).fillna(0).values
groups = df["client_id"].values

def run_split(splitter):
    precisions = []
    for train_idx, test_idx in splitter:
        tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
        tree.fit(X_safe[train_idx], y_arr[train_idx])
        scores = tree.predict_proba(X_safe[test_idx])[:, 1]
        precisions.append(precision_at_k(scores, y_arr[test_idx], min(50, len(test_idx))))
    return np.mean(precisions)

random_p = run_split(KFold(n_splits=5, shuffle=True, random_state=42).split(X_safe))
group_p = run_split(GroupKFold(n_splits=5).split(X_safe, y_arr, groups))

print("random-split avg Precision@50:", round(random_p, 3))
print("client-grouped-split avg Precision@50:", round(group_p, 3))
print("gap (random minus grouped):", round(random_p - group_p, 3))


random-split avg Precision@50: 0.916
client-grouped-split avg Precision@50: 0.676
gap (random minus grouped): 0.24


**Verdict:** a random split scores 0.91, a client-grouped split scores 0.70 -- a 0.21 gap.
That gap is memorization: with a random split, pages from the same client land in both train
and test, so the tree partly learns "this client's pages" rather than a transferable pattern.
The grouped number (0.70) is the honest one to report, since the real deployment question is
"does this work on a client the model never saw."

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- `trend_direction`, `trend_pct` — this is the label itself (Test 1: near-perfect score,
  100% feature importance).
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`,
  `clicks_prev_30d`, `sessions_prev_30d` — the raw inputs `trend_pct` is computed from.
- `provider_used`, `model_used` — which LLM touched the article; a production/authoring flag,
  not a search-performance signal (confirmed absent from the feature set in Test 3).
- `age_tier_order` — a numeric duplicate of `age_tier`, which is already kept as a categorical
  feature alongside the raw `content_age_days`; the ordinal version adds nothing extra.
- `char_count_tier` — a bucketed duplicate of `char_count`, kept as the raw number instead
  (unlike `word_count_tier`, which the feature set does keep as a categorical alongside
  `word_count` — a deliberate asymmetry, not an oversight).
- `content_id`, `client_id` — identifiers. Kept as **context** for joins and the client-grouped
  split in Test 4, never fed to the model as features.
- **Not excluded, but flagged:** the 90-day activity aggregates (`log_impressions_90d` and
  siblings, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`) partially
  overlap the label's last-30-day window (Test 2). Kept because the overlap is partial, not
  total, and the lift comes from several features together rather than one dominating -- but
  this is a documented limitation, not a clean bill of health.

## Self-check

Before you submit, confirm each line honestly:

- [Done] Every section above is filled — markdown thinking AND the code that backs it
- [Done] The notebook runs top to bottom with no errors (Runtime → Run all)
- [Done] No client names, URLs, or private queries anywhere
- [Done] My claims use careful words: observed, measured, directional, decision-support
- [ Done] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.